# Data Profile & Data Quality 주입 효과 비교 분석 및 검증

이 노트북은 데이터 가버넌스 및 메타데이터 관리(Data Profile 및 Data Quality)가 대화형 분석 에이전트(LLM)의 SQL 작성 성능 및 분석 결과 신뢰도에 미치는 실질적인 효과를 입증하고 검증합니다.

동일한 지시어 프롬프트를 유지한 상태에서 오직 **컨텍스트(메타데이터)의 유무**에 따라 에이전트가 데이터 형상을 파악하여 작성하는 SQL 결과의 정합성 격차를 대조합니다.

### 학습 목표
1. **메타데이터 가치 증명**: 데이터 거버넌스(프로파일/품질) 정보 유무가 AI 에이전트의 SQL 생성 신뢰도에 미치는 영향을 검증합니다.
2. **컨텍스트 기반 SQL 정확도 향상**: 실제값 분포(예: `'Brasil'`) 정보를 에이전트에게 함께 주입하여 값 매핑의 오작동을 예방하는 기법을 실증합니다.
3. **방어적 SQL 작성 유도**: 비정상적 오염 데이터 상태(예: 음수 판매가) 정보를 감지한 에이전트가 자동적으로 예외 처리 필터(`WHERE` 절)를 가미하는지 확인합니다.

### 작업 파이프라인 및 분석 시나리오

1. **환경 초기화**: Gemini(Google GenAI SDK) 및 BigQuery 클라이언트를 초기화하고 REST API 통신 환경을 구축합니다.
2. **테스트 데이터셋 샌드박스 생성**: 분석용 원본 데이터를 보호하고 격리된 시뮬레이션을 수행하기 위해 복제 테스트 테이블을 생성합니다.
3. **데이터 프로파일(Data Profile) 효용성 실증 (값 매핑 능력)**:
   * **시나리오**: `users` 테이블에서 "국가가 브라질인 고객은 총 몇 명인가요?" 질의 시, DB 내 실제 국가는 영어 스펠링 `'Brazil'`이 아닌 포르투갈어인 `'Brasil'`로 저장되어 있는 특이점을 파악해야 합니다.
   * **검증**: 에이전트에게 테이블 정보만 준 경우와 데이터 프로파일(값 분포 정보)을 함께 전달한 경우의 SQL 작성 정확도(`'Brasil'` vs `'Brazil'`)를 비교 분석합니다.
4. **데이터 품질(Data Quality) 효용성 실증 (방어적 쿼리 작성 능력)**:
   * **시나리오**: `products` 테이블에서 "상품들의 평균 판매가를 구하는 SQL을 작성해주세요" 질의 시, 특정 상품 가격 데이터를 `-9999.0`으로 인위적으로 손상시킴.
   * **검증**: 동일한 지시문 하에서 품질 실패 정보(`retail_price > 0` 룰 위반)를 확인한 에이전트가 스스로 방어적 조건절(`WHERE retail_price > 0`)을 추가로 작성하는지 대조합니다.
5. **샌드박스 자원 정리**: 검증 과정에 사용된 임시 테스트 데이터셋 및 DataScan 리소스를 정리합니다.

## Step 1: 환경 초기화

이 단계에서는 분석 검증에 필요한 Google Cloud 클라이언트 SDK 및 GenAI(Vertex AI) 라이브러리를 임포트하고, 프로젝트 실행을 위한 Application Default Credentials(ADC) 기반 인증 환경을 설정합니다.

> **[참고]**
> **인증 토큰 발급**: Knowledge Catalog DataScan REST API를 호출하기 위해 GCP 사용자/서비스 계정의 Access Token을 발급받아 API 요청 헤더에 활용합니다. 또한 Gemini SDK 호출을 위해 `genai.Client`를 초기화합니다.

In [ ]:
import json
import time
import ssl
import urllib.request
import google.auth
from google.auth.transport.requests import Request
from google.cloud import bigquery
from google import genai

# 1. 환경 설정 (Application Default Credentials 사용)
credentials, PROJECT_ID = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
TEST_DATASET_ID = "thelook_ecommerce_test"  # 테스트 데이터셋
LOCATION = "asia-northeast3"

# 2. REST API 호출용 인증 토큰 발급
credentials.refresh(Request())
ACCESS_TOKEN = credentials.token

ssl_context = ssl._create_unverified_context()
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
genai_client = genai.Client(vertexai=True)

print(f"Project: {PROJECT_ID}, Test Dataset: {TEST_DATASET_ID}")
print("환경 준비 완료!")

## Step 2: API 호출 공통 함수 정의

Knowledge Catalog DataScan API 호출을 수행하기 위한 HTTP REST 요청 처리 유틸리티와 데이터 스캔 작업을 모니터링하고 결과를 획득하는 헬퍼 함수를 정의합니다. 또한, 4단계와 5단계 분석 검증 코드의 가독성을 높이기 위한 SQL 생성 및 실행 헬퍼 함수도 포함하고 있습니다.

> **[참고]**
> **REST API 사용**: Knowledge Catalog SDK 대신 직접 HTTP Request 방식을 구현하여 리전(Location) 설정 등의 유연성을 확보하고 DataScan Job 결과를 수집합니다.

In [ ]:
def make_rest_request(url, method="GET", body_dict=None, max_retries=5):
    req = urllib.request.Request(url, method=method)
    req.add_header("Authorization", f"Bearer {ACCESS_TOKEN}")
    req.add_header("Content-Type", "application/json")
    data = None
    if body_dict:
        data = json.dumps(body_dict).encode("utf-8")
    
    retries = 0
    backoff = 2  # 시작 대기 시간 (초)
    
    while True:
        try:
            with urllib.request.urlopen(req, data=data, context=ssl_context) as response:
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            err_msg = e.read().decode("utf-8")
            if e.code == 429 and retries < max_retries:
                print(f"  [429 Quota Exceeded] {backoff}초 후 재시도 합니다 (재시도: {retries + 1}/{max_retries})...")
                time.sleep(backoff)
                retries += 1
                backoff *= 2
                continue
            raise Exception(f"HTTP {e.code} - {err_msg}")

def get_or_create_data_profile_scan(scan_id, source_table_resource, sampling_percent=10.0):
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    try:
        make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Profile Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e) or "does not exist" in str(e):
            print(f"  -> 새로운 Data Profile Scan 생성 중: {scan_id}...")
            create_url = f'https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}'
            body = {
                "data": {
                    "resource": source_table_resource
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_PROFILE",
                "dataProfileSpec": {
                    "samplingPercent": sampling_percent,
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    break
                time.sleep(5)
            print(f"  -> Data Profile Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

def get_or_create_data_quality_scan(scan_id, source_table_resource, sampling_percent=100.0):
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    try:
        make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Quality Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e) or "does not exist" in str(e):
            print(f"  -> 새로운 Data Quality Scan 생성 중: {scan_id}...")
            create_url = f'https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}'
            body = {
                "data": {
                    "resource": source_table_resource
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_QUALITY",
                "dataQualitySpec": {
                    "rules": [
                        {
                            "column": "retail_price",
                            "dimension": "VALIDITY",
                            "rangeExpectation": {
                                "minValue": "0",
                                "strictMinEnabled": True
                            },
                            "description": "Retail price must be greater than 0"
                        }
                    ],
                    "samplingPercent": sampling_percent,
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    break
                time.sleep(5)
            print(f"  -> Data Quality Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

def run_datascan_and_wait(scan_id):
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    print(f"  -> DataScan 실행 요청 중...")
    run_res = make_rest_request(run_url, method="POST")
    job_name = run_res["job"]["name"]
    job_url = f"https://dataplex.googleapis.com/v1/{job_name}?view=FULL"
    while True:
        job = make_rest_request(job_url, method="GET")
        state = job.get("state")
        print(f"     [폴링] 현재 상태: {state}")
        if state == "SUCCEEDED":
            return job
        elif state in ["FAILED", "CANCELLED"]:
            raise Exception(f"DataScan Job 오류 발생: {state}")
        time.sleep(10)

def get_latest_job_details(scan_id):
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}/jobs"
    jobs = make_rest_request(get_url, method="GET")
    if not jobs.get("dataScanJobs"):
        return None
    latest_job_name = jobs.get("dataScanJobs", [])[0]["name"]
    job_url = f"https://dataplex.googleapis.com/v1/{latest_job_name}?view=FULL"
    return make_rest_request(job_url, method="GET")

# [공통 유틸리티] LLM을 사용해 SQL 자동 생성
def generate_sql_for_question(question, schema_desc):
    prompt = f"""
당신은 빅쿼리(BigQuery) 전문가입니다. 다음 질문에 답변할 수 있는 SQL 쿼리를 작성해 주세요:
"{question}"

오직 아래에 제공된 테이블 스키마 정보만을 사용하여 쿼리를 작성해 주세요. (프로젝트 및 데이터셋 식별자가 포함된 정확한 테이블명을 사용해야 합니다):
{schema_desc}

다음 규칙을 엄격히 준수해 주세요:
* 데이터셋의 실제 데이터 값이나 통계 정보에 대해 사전 학습된 지식이나 외부 지식을 절대 가정하거나 사용하지 마세요. 오직 제공된 스키마 정의, 컬럼 프로파일 및 통계 정보에만 의존하여 쿼리를 작성해야 합니다.

답변에는 ```sql과 ```로 감싸인 SQL 코드만 반환해 주세요. 다른 설명은 포함하지 마세요.
"""
    response = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return response.text.replace("```sql", "").replace("```", "").strip()

# [공통 유틸리티] 빅쿼리 쿼리 실행 및 결과 출력
def execute_and_print_query(label, sql, expected_unit="명"):
    print(f"\n[실행] {label}")
    print(f"Generated SQL:\n{sql}")
    try:
        query_job = bq_client.query(sql)
        results = list(query_job.result())
        
        if not results:
            print("  -> 결과 없음")
            return 0
            
        val = results[0][0]
        if isinstance(val, float):
            print(f"  -> 결과값: {val:.4f} {expected_unit}")
        else:
            print(f"  -> 결과값: {val} {expected_unit}")
        return val
    except Exception as e:
        print(f"  -> 쿼리 실행 실패: {e}")
        return None

## Step 3: 테스트 데이터셋 샌드박스 생성

테스트 데이터 전용 데이터셋(`thelook_ecommerce_test`)을 내 프로젝트에 생성하고, BigQuery 공개 데이터셋의 `users` 테이블과 `products` 테이블을 이 위치로 복제합니다.

> **[참고]**
> **검증용 격리 환경 구축**: 원본 데이터셋에 영향을 주지 않으면서 음수 데이터 오염 등의 테스트를 자유롭게 수행하기 위해, 별도의 테스트 데이터셋(`thelook_ecommerce_test`)을 생성하고 분석 대상 테이블을 복제합니다.

In [ ]:
# A. 테스트용 데이터셋 생성
dataset_ref = bigquery.DatasetReference(PROJECT_ID, TEST_DATASET_ID)
try:
    bq_client.get_dataset(dataset_ref)
    print(f"데이터셋이 이미 존재합니다: {TEST_DATASET_ID}")
except Exception:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = LOCATION
    bq_client.create_dataset(dataset)
    print(f"신규 데이터셋 생성 완료: {TEST_DATASET_ID}")

# B. users 테이블 복제
source_table_users = "bigquery-public-data.thelook_ecommerce.users"
dest_table_users = f"{PROJECT_ID}.{TEST_DATASET_ID}.users"
copy_job_users = bq_client.copy_table(source_table_users, dest_table_users)
copy_job_users.result()
print("users 테이블 복제 완료.")

# C. products 테이블 복제
source_table_products = "bigquery-public-data.thelook_ecommerce.products"
dest_table_products = f"{PROJECT_ID}.{TEST_DATASET_ID}.products"
copy_job_products = bq_client.copy_table(source_table_products, dest_table_products)
copy_job_products.result()
print("products 테이블 복제 완료.")

## Step 4: 데이터 프로파일(Data Profile) 효용성 실증 (값 매핑 능력)

"국가가 브라질인 고객은 총 몇 명인가요?" 라는 질문에 대해 복제된 `users` 테이블을 대상으로 데이터 프로파일 정보 제공 유무에 따른 두 시나리오의 SQL 코드 및 실행 결과를 비교하여 메타데이터가 미치는 실질적 효용성을 검증합니다.

> **[참고]**
> **검증 포인트**:
> 1. **시나리오 A (메타데이터 없음)**: 스키마 컬럼 정보만으로 쿼리를 작성하여, 실제 데이터 표기 방식(`'Brasil'`)을 알지 못해 영문명인 `'Brazil'`로 조건절을 적용(결과 0명)하는 실수를 관찰합니다.
> 2. **시나리오 B (메타데이터 포함)**: 신규 데이터 프로파일을 통해 추출된 컬럼 내 다빈도 실제 값 분포(`'Brasil'`)를 학습 스키마 컨텍스트로 제공받아, LLM이 물리 데이터를 정확히 식별하여 올바른 쿼리(`'Brasil'`)를 빌드하는지 검증합니다.

In [ ]:
scan_id_prof = "dp-thelook-test-users"
local_table_resource = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{TEST_DATASET_ID}/tables/users"

# A. 새로운 프로파일 스캔 리소스 생성 및 실행 (10% 샘플링 적용)
get_or_create_data_profile_scan(scan_id_prof, local_table_resource, sampling_percent=10.0)

print("데이터 프로파일 스캔 실행...")
prof_job = run_datascan_and_wait(scan_id_prof)

users_profile = prof_job.get("dataProfileResult", {})
local_table_users = bq_client.get_table(bq_client.dataset(TEST_DATASET_ID).table("users"))
public_table_users = bq_client.get_table("bigquery-public-data.thelook_ecommerce.users")

# B. 빅쿼리 테이블 메타데이터 라벨 반영 (dataplex- 공식 접두사 적용)
labels = dict(local_table_users.labels or {})
labels["dataplex-data-profile-published-scan"] = scan_id_prof
labels["dataplex-data-profile-published-project"] = PROJECT_ID
labels["dataplex-data-profile-published-location"] = LOCATION
local_table_users.labels = labels
bq_client.update_table(local_table_users, ["labels"])
print(f"테이블 {local_table_users.table_id}에 프로파일 퍼블리싱 라벨 적용 완료!")

# C. [시나리오 A] 내 복제 테스트 테이블 기본 스키마 정보 구성 (프로파일 통계 제외)
schema_a = f"Table: `{PROJECT_ID}.{TEST_DATASET_ID}.users`\nColumns:\n"
for f in local_table_users.schema:
    schema_a += f" - {f.name}: {f.field_type} ({f.mode})\n"

# D. [시나리오 B] 내 복제 테스트 테이블 + 데이터 프로파일 정보 구성
def build_rich_table_schema(table, profile_data):
    schema_desc = f"Table: `{PROJECT_ID}.{TEST_DATASET_ID}.{table.table_id}`\nColumns:\n"
    profile_fields = {f["name"]: f.get("profile", {}) for f in profile_data.get("profile", {}).get("fields", [])}
    
    for f in table.schema:
        schema_desc += f" - {f.name}: {f.field_type} ({f.mode})\n"
        prof = profile_fields.get(f.name, {})
        if prof:
            null_ratio = prof.get("nullRatio", 0) * 100
            distinct_ratio = prof.get("distinctRatio", 0) * 100
            schema_desc += f"   * Profile Stats: Null Ratio={null_ratio:.1f}%, Distinct Ratio={distinct_ratio:.1f}%\n"
            if "topNValues" in prof:
                top_vals = [v.get("value") for v in prof["topNValues"]]
                schema_desc += f"     - Common Values (Top N): {top_vals}\n"
    return schema_desc

schema_b = build_rich_table_schema(local_table_users, users_profile)
question_1 = "국가가 브라질인 고객은 총 몇 명인가요?"

# E. SQL 생성
sql_a = generate_sql_for_question(question_1, schema_a)
sql_b = generate_sql_for_question(question_1, schema_b)

# F. 쿼리 실행
val_a = execute_and_print_query("시나리오 A (메타데이터 없음) - 테스트 데이터셋 쿼리 실행", sql_a)
val_b = execute_and_print_query("시나리오 B (메타데이터 포함) - 테스트 데이터셋 쿼리 실행", sql_b)

# G. 결과 대조 검증 요약 출력
print(
    "\n\n=== [프로파일 분석 검증 요약] ===\n"
    "💡 분석 설명:\n"
    f"1. [시나리오 A] 결과 (메타데이터 누락): {val_a} 명\n"
    f"2. [시나리오 B] 결과 (프로파일 데이터 포함): {val_b} 명\n"
    "\n"
    "3. 원인 분석:\n"
    "   - [시나리오 A]는 단순히 국가명을 일반 표준 영문인 'Brazil'로 유추하여 쿼리(country = 'Brazil')를 자동 작성합니다. 하지만 실제 테이블에 등록된 국가명은 포르투갈어 철자인 'Brasil'이기 때문에 데이터가 매칭되지 않아 0명이 산출됩니다.\n"
    "   - [시나리오 B]는 Knowledge Catalog가 수집한 데이터 프로파일 결과(Common Values (Top N): ['Brasil', ...])를 통해 실제 데이터 값을 확인하고, 이를 활용하여 쿼리(country = 'Brasil')를 정확히 작성해 내어 정상적인 결과를 얻게 됩니다."
)

## Step 5: 데이터 품질(Data Quality) 효용성 실증 (방어적 쿼리 작성 능력)

"상품들의 평균 판매가(retail_price)를 구하는 SQL을 작성해주세요." 라는 질문에 대해, 동일한 오염 데이터 테이블(`{PROJECT_ID}.{TEST_DATASET_ID}.products`)을 타겟으로 품질 정보 제공 유무에 따른 두 시나리오의 SQL 코드 및 실제 실행 결과 평균값 차이를 비교해 메타데이터의 효용성을 검증합니다.

> **[참고]**
> **검증 포인트**:
> 1. **시나리오 A (품질 정보 누락)**: 비정상 데이터(판매가 -9999.0)의 혼입 상태를 모르는 상태에서 일반적인 평균값 쿼리(`AVG(retail_price)`)를 가동하여, 노이즈 가격 데이터로 인해 평균 가격이 극도로 왜곡되는 현상을 목격합니다.
> 2. **시나리오 B (품질 정보 포함)**: `retail_price > 0` 유효성 규칙 실패 경고와 통계 수치를 사전에 제공받은 LLM이 스스로 방어 코드 조건절(`WHERE retail_price > 0`)을 추가하여 신뢰할 수 있는 정확한 통계치를 계산해 내는지 대조하여 검증합니다.

In [ ]:
# A. 판매가 데이터를 음수로 변경하여 데이터 오염
print("음수 판매가 데이터 주입 중...")
update_query = f"""
UPDATE `{PROJECT_ID}.{TEST_DATASET_ID}.products`
SET retail_price = -9999.0
WHERE id = 1
"""
bq_client.query(update_query).result()
print("데이터 오염 완료.")

# API 토큰 재갱신
credentials.refresh(Request())
ACCESS_TOKEN = credentials.token

# B. 품질 스캔 리소스 생성 및 실행 (10% 샘ล링 적용)
scan_id_qual = "dq-thelook-test-products"
local_products_resource = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{TEST_DATASET_ID}/tables/products"

get_or_create_data_quality_scan(scan_id_qual, local_products_resource, sampling_percent=10.0)

print("데이터 품질 스캔 실행...")
qual_job = run_datascan_and_wait(scan_id_qual)

products_quality = qual_job.get("dataQualityResult", {})
print("데이터 품질 검사 통과 여부:", products_quality.get("passed"))
local_table_products = bq_client.get_table(bq_client.dataset(TEST_DATASET_ID).table("products"))

# C. 빅쿼리 테이블 메타데이터 품질 라벨 반영 (dataplex- 공식 접두사 적용)
labels_prod = dict(local_table_products.labels or {})
labels_prod["dataplex-data-quality-published-scan"] = scan_id_qual
labels_prod["dataplex-data-quality-published-project"] = PROJECT_ID
labels_prod["dataplex-data-quality-published-location"] = LOCATION
local_table_products.labels = labels_prod
bq_client.update_table(local_table_products, ["labels"])
print(f"테이블 {local_table_products.table_id}에 품질 퍼블리싱 라벨 적용 완료!")

# D. [시나리오 A] 내 오염 테이블 타겟팅하되, 품질 정보는 누락한 베이직 스키마 구성
schema_a_local = f"Table: `{PROJECT_ID}.{TEST_DATASET_ID}.products`\nColumns:\n"
for f in local_table_products.schema:
    schema_a_local += f" - {f.name}: {f.field_type} ({f.mode})\n"

# E. [시나리오 B] 내 오염 테이블 타겟팅 + 데이터 품질 진단 실패 통계 결합 스키마 구성
schema_b_quality = f"Table: `{PROJECT_ID}.{TEST_DATASET_ID}.products`\nColumns:\n"
for f in local_table_products.schema:
    schema_b_quality += f" - {f.name}: {f.field_type} ({f.mode})\n"

schema_b_quality += "\nData Quality Rules and Status:\n"
for rule in products_quality.get("rules", []):
    rule_desc = rule.get("rule", {}).get("description", "")
    passed = rule.get("passed")
    evaluated = rule.get("evaluatedCount")
    passed_cnt = rule.get("passedCount")
    schema_b_quality += f" - Rule: {rule_desc} -> PASSED={passed} (Passed Count={passed_cnt}/{evaluated})\n"
    if not passed:
        schema_b_quality += f"   * WARNING: This rule failed. Some records violate this constraint (Negative values detected).\n"

question_2 = "상품들의 평균 판매가(retail_price)를 구하는 SQL을 작성해주세요."

# F. SQL 생성
sql_a2 = generate_sql_for_question(question_2, schema_a_local)
sql_b2 = generate_sql_for_question(question_2, schema_b_quality)

# G. 쿼리 실행
val_a2 = execute_and_print_query("시나리오 A (품질 정보 누락) - 오염 테이블 대상 평균값 조회", sql_a2, expected_unit="달러")
val_b2 = execute_and_print_query("시나리오 B (품질 정보 포함) - 오염 테이블 대상 평균값 조회", sql_b2, expected_unit="달러")

# H. 결과 대조 검증 요약 출력
print(
    "\n\n=== [품질 분석 검증 요약] ===\n"
    "💡 분석 설명:\n"
    f"1. [시나리오 A] 결과 (품질 정보 누락): {val_a2:.4f} 달러\n"
    f"2. [시나리오 B] 결과 (품질 정보 포함): {val_b2:.4f} 달러\n"
    "\n"
    "3. 원인 분석:\n"
    "   - [시나리오 A]는 내 테스트 테이블을 타겟으로 쿼리하지만, 데이터 품질 진단 경고가 누락되어 단순 AVG(retail_price) 쿼리를 작성합니다. 그 결과 오염 데이터(-9999.0 달러)가 합산되어 평균 가격이 극단적으로 낮게 왜곡됩니다.\n"
    "   - [시나리오 B]는 동일한 테스트 테이블을 타겟으로 하지만 'Retail price must be greater than 0' 규격 위반 경고(PASSED=False)를 LLM이 스키마 정보에서 확인하므로, 쿼리에 자동으로 `WHERE retail_price > 0` 방어 필터를 추가합니다. 이에 따라 오염 데이터가 격리된 올바른 정상 평균 판매가격이 구해집니다."
)